# HTF-1 SPR Steady-State Affinity Fitting Notebook

专为 Na/K-ATPase α1 与 HTF-1 亲和力数据设计的稳态Langmuir模型拟合。
直接运行即可输出 Kd、Rmax、图表和报告。

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from scipy.stats import t

# ================== 参数区 ==================
csv_file = "HTF-1_SPR_steady_state.csv"   # ←←← 请替换为您的Biacore导出CSV文件路径
alpha = 0.05                               # 95%置信区间

# Langmuir模型
def langmuir(x, Rmax, Kd):
    return Rmax * x / (Kd + x)

# ================== 1. 加载数据 ==================
df = pd.read_csv(csv_file)
conc = df['Conc_nM'].values.astype(float)   # 单位 nM
req = df['Req_RU'].values.astype(float)     # 稳态响应 RU

print('数据加载成功！')
print(df.head())

In [ ]:
# ================== 2. 曲线拟合 ==================
p0 = [np.max(req)*1.1, 50]                  # 初始猜测
popt, pcov = curve_fit(langmuir, conc, req, p0=p0, bounds=(0, np.inf))

Rmax, Kd = popt
perr = np.sqrt(np.diag(pcov))               # 标准误差

# 95%置信区间
dof = len(conc) - 2
tval = t.ppf(1 - alpha/2, dof)
ci_Rmax = tval * perr[0]
ci_Kd = tval * perr[1]

print(f"Rmax = {Rmax:.2f} ± {ci_Rmax:.2f} RU")
print(f"Kd   = {Kd:.2f} ± {ci_Kd:.2f} nM (目标 < 100 nM)")

In [ ]:
# ================== 3. 绘图 ==================
x_fit = np.logspace(np.log10(min(conc)*0.5), np.log10(max(conc)*2), 200)
y_fit = langmuir(x_fit, *popt)

plt.figure(figsize=(8, 6))
plt.scatter(conc, req, color='blue', label='实验数据 (Biacore)')
plt.plot(x_fit, y_fit, 'r-', label=f'Langmuir拟合曲线\nKd = {Kd:.2f} nM')
plt.xscale('log')
plt.xlabel('HTF-1 浓度 (nM)')
plt.ylabel('稳态响应 (RU)')
plt.title('HTF-1 / Na+/K+-ATPase α1 稳态亲和力拟合')
plt.legend()
plt.grid(True, which='both', ls='--')
plt.savefig('HTF-1_SPR_fitting.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ================== 4. 保存结果 ==================
result = pd.DataFrame({
    'Parameter': ['Rmax', 'Kd'],
    'Value': [Rmax, Kd],
    'StdErr': perr,
    '95% CI lower': [Rmax - ci_Rmax, Kd - ci_Kd],
    '95% CI upper': [Rmax + ci_Rmax, Kd + ci_Kd]
})
result.to_csv('HTF-1_SPR_fitting_results.csv', index=False)
print("\n拟合完成！结果已保存：")
print(result)
print("\n图片保存为 HTF-1_SPR_fitting.png")

## 使用说明
1. 将Biacore稳态数据导出为 CSV（列名必须为 `Conc_nM` 和 `Req_RU`）
2. 修改第1个cell中的 `csv_file` 路径
3. 依次运行所有cell
4. 输出自动生成 Kd 值、拟合图和 CSV 报告（直接用于IND）